- Use a strong baseline first
- TF IDF with unigrams + bigrams
- Logistic Regression
- Stratified split
- Weighted metrics
- Save vectorizer + model

- Section 1  Load cleaned training data
- Section 2  Final target setup
- Section 3  Train test split
- Section 4  TF IDF vectorization
- Section 5  Train Models
- Section 6  Evaluate
- Section 7  Save artifacts
- Section 8  Write training summary

In [19]:
# Import data handling libraries
import pandas as pd
import numpy as np

# Import text cleaning support
import re

# Import plotting libraries for quick checks if needed
import matplotlib.pyplot as plt
import seaborn as sns

# Import TF IDF vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Import baseline text classification models
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

# Import evaluation metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Import model saving utility
import joblib

In [2]:
# Show all columns when previewing data
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

In [3]:
# Load sampled training data from csv
df_train = pd.read_csv("../data/raw/cfpb_sample_500k.csv")

In [4]:
# Check dataset shape
df_train.shape

(500000, 18)

In [5]:
# Preview the first rows
df_train.head()

,date_received,product,sub-product,issue,sub-issue,consumer_complaint_narrative,company_public_response,company,state,zip_code,tags,consumer_consent_provided,submitted_via,date_sent_to_company,company_response_to_consumer,timely_response,consumer_disputed,complaint_id
0,2022-06-24,"Credit reporting, credit repair services, or other personal consumer reports",Credit reporting,Problem with a credit reporting company's investigation into an existing problem,Investigation took more than 30 days,Hi I am submitting this XXXX XXXX this isn't any influence and this is not a third party. XXXX has low and unfair credit number for me in their report. I have 87complained times. The problem has n...,Company has responded to the consumer and the CFPB and chooses not to provide a public response,Experian Information Solutions Inc.,IL,60564,NaN,Consent provided,Web,2022-06-24,Closed with non-monetary relief,Yes,NaN,5702809
1,2025-07-18,Debt collection,Federal student loan debt,Attempts to collect debt not owed,Debt was result of identity theft,"A federal education loan was fraudulently opened in my name. I filed an FTC Identity Theft Report, a DOE Loan Forgery Discharge Application, and submitted supporting documents. Despite this, the D...",NaN,Servicer under contract with Federal Student Aid,CA,95682,NaN,Consent provided,Web,2025-07-18,Untimely response,No,NaN,14736409
2,2025-07-10,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,"I am writing to have the following information removed from my credit file, the items that I need deleted are going to be attached in a word document. I am a victim of identity theft. I have multi...",NaN,"EQUIFAX, INC.",OH,432XX,NaN,Consent provided,Web,2025-07-10,Closed with non-monetary relief,Yes,NaN,14558425
3,2023-04-26,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a credit reporting company's investigation into an existing problem,Was not notified of investigation status or results,"I want this late payment remove from my account, and I never been late with this account I working so hard to pay this monthly and this is not acceptable. \nSee the documents attached.",Company has responded to the consumer and the CFPB and chooses not to provide a public response,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",FL,33404,NaN,Consent provided,Web,2023-04-26,Closed with non-monetary relief,Yes,NaN,6895444
4,2025-11-19,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,I am disputing several items on my Experian credit report that appear to be inaccurate or incomplete. The details of the accounts in question are as follows : 1. Charged-Off Accounts XXXX XXXX XXX...,Company has responded to the consumer and the CFPB and chooses not to provide a public response,Experian Information Solutions Inc.,NC,XXXXX,NaN,Consent provided,Web,2025-11-19,Closed with explanation,Yes,NaN,17362554


In [6]:
# Check original product distribution
df_train["product"].value_counts()

product
Credit reporting or other personal consumer reports                             222115
Credit reporting, credit repair services, or other personal consumer reports    107990
Debt collection                                                                  55063
Checking or savings account                                                      22738
Mortgage                                                                         18835
Money transfer, virtual currency, or money service                               15097
Credit card                                                                      14877
Credit card or prepaid card                                                      14498
Student loan                                                                      8064
Vehicle loan or lease                                                             6497
Credit reporting                                                                  4270
Payday loan, title loan, or persona

In [7]:
# Map similar product categories into unified groups
product_map = {
    "Credit reporting, credit repair services, or other personal consumer reports": "Credit reporting",
    "Credit reporting or other personal consumer reports": "Credit reporting",
    "Credit reporting": "Credit reporting",

    "Credit card or prepaid card": "Credit card",
    "Prepaid card": "Credit card",
    "Credit card": "Credit card",

    "Checking or savings account": "Bank account",
    "Bank account or service": "Bank account",

    "Money transfer, virtual currency, or money service": "Money transfer",
    "Money transfers": "Money transfer",
    "Virtual currency": "Money transfer",

    "Vehicle loan or lease": "Auto loan",

    "Payday loan, title loan, or personal loan": "Personal loan",
    "Payday loan, title loan, personal loan, or advance loan": "Personal loan",
    "Payday loan": "Personal loan",

    "Consumer Loan": "Personal loan"
}

In [8]:
# Create cleaned product target column
df_train["product_clean"] = df_train["product"].replace(product_map)

In [10]:
# Check cleaned product distribution
df_train["product_clean"].value_counts()

product_clean
Credit reporting             334375
Debt collection               55063
Credit card                   30796
Bank account                  24679
Mortgage                      18835
Money transfer                15312
Student loan                   8064
Auto loan                      6497
Personal loan                  5720
Debt or credit management       623
Other financial service          36
Name: count, dtype: int64

In [ ]:
# A simple text cleaning function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [12]:
# Keep only the text and cleaned target columns
df_train = df_train[["consumer_complaint_narrative", "product_clean"]].copy()

In [13]:
# Rename columns for training clarity
df_train.columns = ["text", "target"]

In [14]:
df_train

,text,target
0,Hi I am submitting this XXXX XXXX this isn't any influence and this is not a third party. XXXX has low and unfair credit number for me in their report. I have 87complained times. The problem has n...,Credit reporting
1,"A federal education loan was fraudulently opened in my name. I filed an FTC Identity Theft Report, a DOE Loan Forgery Discharge Application, and submitted supporting documents. Despite this, the D...",Debt collection
2,"I am writing to have the following information removed from my credit file, the items that I need deleted are going to be attached in a word document. I am a victim of identity theft. I have multi...",Credit reporting
3,"I want this late payment remove from my account, and I never been late with this account I working so hard to pay this monthly and this is not acceptable. \nSee the documents attached.",Credit card
4,I am disputing several items on my Experian credit report that appear to be inaccurate or incomplete. The details of the accounts in question are as follows : 1. Charged-Off Accounts XXXX XXXX XXX...,Credit reporting
...,...,...
499995,XXXX should not be reporting to XXXX ; XXXX and Transunion that I am ( 30 ) days late on the XXXX partial account number XXXX. ( Please see page attached from my credit report. ) This account had...,Credit reporting
499996,The credit bureaus are reporting inaccurate/outdated/incomplete personal information.,Credit reporting
499997,"We receive at least 3 calls an hour, all day long, all hours of the day, even weekends. Have asked them to stop and they have not. The dates have been ongoing for weeks. Todays date, XX/XX/19 was ...",Debt collection
499998,"These can be combined On my credit report, you have shown incorrect accounts that should not be there at all. This is not only unjust to me, but it's also concerning, because I've never done or ma...",Credit reporting


In [15]:
# Apply text cleaning to the complaint text
df_train["text_clean"] = df_train["text"].apply(clean_text)

In [16]:
df_train

,text,target,text_clean
0,Hi I am submitting this XXXX XXXX this isn't any influence and this is not a third party. XXXX has low and unfair credit number for me in their report. I have 87complained times. The problem has n...,Credit reporting,hi i am submitting this xxxx xxxx this isn t any influence and this is not a third party xxxx has low and unfair credit number for me in their report i have 87complained times the problem has not ...
1,"A federal education loan was fraudulently opened in my name. I filed an FTC Identity Theft Report, a DOE Loan Forgery Discharge Application, and submitted supporting documents. Despite this, the D...",Debt collection,a federal education loan was fraudulently opened in my name i filed an ftc identity theft report a doe loan forgery discharge application and submitted supporting documents despite this the depart...
2,"I am writing to have the following information removed from my credit file, the items that I need deleted are going to be attached in a word document. I am a victim of identity theft. I have multi...",Credit reporting,i am writing to have the following information removed from my credit file the items that i need deleted are going to be attached in a word document i am a victim of identity theft i have multiple...
3,"I want this late payment remove from my account, and I never been late with this account I working so hard to pay this monthly and this is not acceptable. \nSee the documents attached.",Credit card,i want this late payment remove from my account and i never been late with this account i working so hard to pay this monthly and this is not acceptable see the documents attached
4,I am disputing several items on my Experian credit report that appear to be inaccurate or incomplete. The details of the accounts in question are as follows : 1. Charged-Off Accounts XXXX XXXX XXX...,Credit reporting,i am disputing several items on my experian credit report that appear to be inaccurate or incomplete the details of the accounts in question are as follows 1 charged off accounts xxxx xxxx xxxx xx...
...,...,...,...
499995,XXXX should not be reporting to XXXX ; XXXX and Transunion that I am ( 30 ) days late on the XXXX partial account number XXXX. ( Please see page attached from my credit report. ) This account had...,Credit reporting,xxxx should not be reporting to xxxx xxxx and transunion that i am 30 days late on the xxxx partial account number xxxx please see page attached from my credit report this account had been transfe...
499996,The credit bureaus are reporting inaccurate/outdated/incomplete personal information.,Credit reporting,the credit bureaus are reporting inaccurate outdated incomplete personal information
499997,"We receive at least 3 calls an hour, all day long, all hours of the day, even weekends. Have asked them to stop and they have not. The dates have been ongoing for weeks. Todays date, XX/XX/19 was ...",Debt collection,we receive at least 3 calls an hour all day long all hours of the day even weekends have asked them to stop and they have not the dates have been ongoing for weeks todays date xx xx 19 was the las...
499998,"These can be combined On my credit report, you have shown incorrect accounts that should not be there at all. This is not only unjust to me, but it's also concerning, because I've never done or ma...",Credit reporting,these can be combined on my credit report you have shown incorrect accounts that should not be there at all this is not only unjust to me but it s also concerning because i ve never done or made a...


In [17]:
# Check for missing values
df_train.isnull().sum()

text          0
target        0
text_clean    0
dtype: int64

### Train Test Split

In [18]:
# Split features and target
X = df_train["text_clean"]
y = df_train["target"]

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size= 0.2,
    random_state=42,
    stratify=y
)

In [29]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((400000,), (100000,), (400000,), (100000,))

In [30]:
# Check target distribution in training set
y_train.value_counts(normalize=True)

target
Credit reporting             0.668750
Debt collection              0.110125
Credit card                  0.061593
Bank account                 0.049357
Mortgage                     0.037670
Money transfer               0.030625
Student loan                 0.016127
Auto loan                    0.012995
Personal loan                0.011440
Debt or credit management    0.001245
Other financial service      0.000073
Name: proportion, dtype: float64

In [31]:
# Check target distribution in training set
y_train.value_counts(normalize=True)

target
Credit reporting             0.668750
Debt collection              0.110125
Credit card                  0.061593
Bank account                 0.049357
Mortgage                     0.037670
Money transfer               0.030625
Student loan                 0.016127
Auto loan                    0.012995
Personal loan                0.011440
Debt or credit management    0.001245
Other financial service      0.000073
Name: proportion, dtype: float64

### TF IDF: 
### **TF-IDF Calculations**

**Term Frequency (TF)**  
This measures how frequently a term appears in a document. It is calculated as:  
$$\text{TF}(t, d) = \frac{\text{Number of times term } t \text{ appears in document } d}{\text{Total number of terms in document } d}$$

**Inverse Document Frequency (IDF)**  
This measures how important a term is across the entire corpus. It is calculated as:  
$$\text{IDF}(t, D) = \log\left( \frac{\text{Total number of documents in corpus } D}{\text{Number of documents containing term } t} \right)$$

**TF-IDF Score**  
The TF-IDF score for a term is the product of its TF and IDF scores:  
$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

In [32]:
# Create TF-IDF vectorizer with unigrams and bigrams
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english"
)

In [33]:
# Fit on training text and transform both train and test text
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [34]:
# Check vectorized matrix shapes
print("X_train_vec shape:", X_train_vec.shape)
print("X_test_vec shape:", X_test_vec.shape)

X_train_vec shape: (400000, 10000)
X_test_vec shape: (100000, 10000)


In [35]:
# Check vocabulary size
print("Vocabulary size:", len(vectorizer.vocabulary_))

Vocabulary size: 10000


In [38]:
vectorizer.vocabulary_

{'received': np.int64(7064),
 'debt': np.int64(2914),
 'collection': np.int64(2121),
 'notification': np.int64(6100),
 'xxxx': np.int64(9530),
 'mail': np.int64(5642),
 'asking': np.int64(1351),
 '17000': np.int64(265),
 '00': np.int64(0),
 'address': np.int64(1033),
 'company': np.int64(2194),
 'called': np.int64(1762),
 'heard': np.int64(4491),
 'today': np.int64(8789),
 'contact': np.int64(2493),
 'trust': np.int64(8918),
 'legitimate': np.int64(5420),
 'collector': np.int64(2147),
 'instead': np.int64(5054),
 'immediately': np.int64(4667),
 'file': np.int64(4008),
 'complaint': np.int64(2229),
 'matter': np.int64(5724),
 'investigated': np.int64(5112),
 'properly': np.int64(6827),
 'received debt': np.int64(7071),
 'debt collection': np.int64(2920),
 'xxxx xxxx': np.int64(9950),
 'xxxx mail': np.int64(9768),
 '17000 00': np.int64(266),
 'company called': np.int64(2196),
 'called xxxx': np.int64(1780),
 'xxxx heard': np.int64(9720),
 'debt collector': np.int64(2922),
 'immediately f

In [36]:
# Preview a few learned terms
list(vectorizer.vocabulary_.keys())[:20]

['received',
 'debt',
 'collection',
 'notification',
 'xxxx',
 'mail',
 'asking',
 '17000',
 '00',
 'address',
 'company',
 'called',
 'heard',
 'today',
 'contact',
 'trust',
 'legitimate',
 'collector',
 'instead',
 'immediately']

### Model Training

#### Logistic Regression

In [39]:
# Create Logistic Regression model
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [40]:
# Train Logistic Regression model
lr_model.fit(X_train_vec, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [41]:
# Make predictions with Logistic Regression
y_pred_lr = lr_model.predict(X_test_vec)

In [42]:
# Check accuracy for Logistic Regression
lr_accuracy = accuracy_score(y_test, y_pred_lr)
print("Logistic Regression Accuracy:", lr_accuracy)

Logistic Regression Accuracy: 0.88958


In [44]:
# Print classification report for Logistic Regression
print(classification_report(y_test, y_pred_lr))

c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


                           precision    recall  f1-score   support

                Auto loan       0.68      0.57      0.62      1299
             Bank account       0.78      0.81      0.79      4936
              Credit card       0.77      0.75      0.76      6159
         Credit reporting       0.93      0.96      0.95     66875
          Debt collection       0.79      0.71      0.75     11013
Debt or credit management       0.82      0.07      0.13       125
           Money transfer       0.85      0.78      0.81      3062
                 Mortgage       0.89      0.88      0.89      3767
  Other financial service       0.00      0.00      0.00         7
            Personal loan       0.63      0.39      0.48      1144
             Student loan       0.85      0.77      0.81      1613

                 accuracy                           0.89    100000
                macro avg       0.73      0.61      0.64    100000
             weighted avg       0.89      0.89      0.89    

c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### SVC

In [45]:
# Create LinearSVC model
svc_model = LinearSVC(random_state=42)

In [46]:
# Train LinearSVC model
svc_model.fit(X_train_vec, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,42


In [47]:
# make predictions
y_pred_svc = svc_model.predict(X_test_vec)

In [48]:
# check accuracy for SVC
svc_accuracy = accuracy_score(y_test, y_pred_svc)
print(f"Linear SVC Accuracy: {svc_accuracy}")

Linear SVC Accuracy: 0.88861


In [49]:
# classification report
print(classification_report(y_test, y_pred_svc))

c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


                           precision    recall  f1-score   support

                Auto loan       0.65      0.56      0.60      1299
             Bank account       0.77      0.81      0.79      4936
              Credit card       0.77      0.74      0.76      6159
         Credit reporting       0.93      0.96      0.95     66875
          Debt collection       0.79      0.71      0.75     11013
Debt or credit management       0.53      0.06      0.11       125
           Money transfer       0.84      0.78      0.81      3062
                 Mortgage       0.88      0.89      0.88      3767
  Other financial service       0.00      0.00      0.00         7
            Personal loan       0.65      0.35      0.45      1144
             Student loan       0.83      0.79      0.81      1613

                 accuracy                           0.89    100000
                macro avg       0.70      0.60      0.63    100000
             weighted avg       0.88      0.89      0.89    

c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### train Multinomial Naive Bayes

In [50]:
# Create Multinomial Naive Bayes model
nb_model = MultinomialNB()

In [51]:
# Train Multinomial Naive Bayes model
nb_model.fit(X_train_vec, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [52]:
# Make predictions with Multinomial Naive Bayes
y_pred_nb = nb_model.predict(X_test_vec)

In [53]:
# Check accuracy for Multinomial Naive Bayes
nb_accuracy = accuracy_score(y_test, y_pred_nb)
print("MultinomialNB Accuracy:", nb_accuracy)

MultinomialNB Accuracy: 0.83794


In [54]:
# Print classification report for Multinomial Naive Bayes
print(classification_report(y_test, y_pred_nb))

c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


                           precision    recall  f1-score   support

                Auto loan       0.57      0.54      0.55      1299
             Bank account       0.63      0.86      0.72      4936
              Credit card       0.54      0.76      0.63      6159
         Credit reporting       0.95      0.90      0.92     66875
          Debt collection       0.68      0.70      0.69     11013
Debt or credit management       0.00      0.00      0.00       125
           Money transfer       0.97      0.43      0.59      3062
                 Mortgage       0.71      0.93      0.80      3767
  Other financial service       0.00      0.00      0.00         7
            Personal loan       0.67      0.17      0.27      1144
             Student loan       0.73      0.79      0.76      1613

                 accuracy                           0.84    100000
                macro avg       0.59      0.55      0.54    100000
             weighted avg       0.86      0.84      0.84    

c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [55]:
# Create a comparison table for all baseline models
results_df = pd.DataFrame({
    "Model": ["Logistic Regression", "LinearSVC", "MultinomialNB"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_svc),
        accuracy_score(y_test, y_pred_nb)
    ],
    "Weighted Precision": [
        precision_score(y_test, y_pred_lr, average="weighted", zero_division=0),
        precision_score(y_test, y_pred_svc, average="weighted", zero_division=0),
        precision_score(y_test, y_pred_nb, average="weighted", zero_division=0)
    ],
    "Weighted Recall": [
        recall_score(y_test, y_pred_lr, average="weighted", zero_division=0),
        recall_score(y_test, y_pred_svc, average="weighted", zero_division=0),
        recall_score(y_test, y_pred_nb, average="weighted", zero_division=0)
    ],
    "Weighted F1": [
        f1_score(y_test, y_pred_lr, average="weighted", zero_division=0),
        f1_score(y_test, y_pred_svc, average="weighted", zero_division=0),
        f1_score(y_test, y_pred_nb, average="weighted", zero_division=0)
    ],
    "Macro F1": [
        f1_score(y_test, y_pred_lr, average="macro", zero_division=0),
        f1_score(y_test, y_pred_svc, average="macro", zero_division=0),
        f1_score(y_test, y_pred_nb, average="macro", zero_division=0)
    ]
})

results_df.sort_values(by="Weighted F1", ascending=False)

,Model,Accuracy,Weighted Precision,Weighted Recall,Weighted F1,Macro F1
0,Logistic Regression,0.88958,0.885634,0.88958,0.886227,0.635319
1,LinearSVC,0.88861,0.884198,0.88861,0.885082,0.627534
2,MultinomialNB,0.83794,0.855286,0.83794,0.838635,0.541284
